<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5512-notebooks/main/_banners/opim5512_banner.svg" width="100%" alt="OPIM 5512 banner"/>


# SHAP the easy way — a **linear** model (rung 1)

**Dr. Dave Wanik · OPIM 5512 · University of Connecticut**

---

Before the tree, meet SHAP where it's **undeniable**. For a linear model
$$f(x)=b+\beta_1 x_1+\beta_2 x_2+\dots$$
a feature's SHAP value collapses to a single line — **no coalitions, no averaging over orders**:
$$\phi_i=\beta_i\,(x_i-\bar x_i)\quad\text{“coefficient} \times \text{how far this row is from average.”}$$
The base value is $f(\bar x)$ = the average prediction, and $\text{base}+\sum_i\phi_i=f(x)$ falls straight out of the equation. We'll compute it by hand and watch the `shap` library return the **same** numbers. Then the companion **`2a_SHAP_ShallowTree_CAHousing`** shows what changes when a model has **interactions**.


## Setup


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
%matplotlib inline

data = fetch_california_housing(as_frame=True).frame
FEATURES = ["MedInc", "AveRooms", "Latitude"]     # same 3 features as the tree notebook
X, y = data[FEATURES], data["MedHouseVal"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)

lin = LinearRegression().fit(Xtr, ytr)
print("Test R2:", round(r2_score(yte, lin.predict(Xte)), 3), " (accuracy isn't the point here — transparency is)")
print("intercept b =", round(lin.intercept_, 4))
for f, b in zip(FEATURES, lin.coef_):
    print(f"  beta[{f:<9}] = {b:+.4f}")

## The base value = the average prediction = $f(\bar x)$

For a linear model the mean of the predictions equals the model evaluated at the mean features — so the SHAP **base value** is just the average prediction.


In [ ]:
xbar = Xtr.mean()                                  # the background 'average' row
base_predict = lin.predict(Xtr).mean()            # average of the predictions
base_fmean   = lin.intercept_ + (lin.coef_ * xbar.values).sum()   # f(mean features)
print("mean of predictions      :", round(base_predict, 4))
print("f(mean features)         :", round(base_fmean, 4), " <- identical for a linear model")

## SHAP by hand: $\phi_i=\beta_i(x_i-\bar x_i)$

Pick one test home. Each feature's contribution is just its coefficient times how far this home sits from the average — one term of the equation.


In [ ]:
x = Xte.iloc[0]
phi = lin.coef_ * (x.values - xbar.values)         # <-- the whole thing
pred = lin.predict(x.to_frame().T)[0]
print("home:", {f: round(float(v),3) for f,v in x.items()})
print(f"base value        = {base_fmean:.4f}")
for f, p in zip(FEATURES, phi):
    print(f"  SHAP[{f:<9}] = {p:+.4f}   ( = beta * (x - mean) )")
print("-"*40)
print(f"base + sum(SHAP)  = {base_fmean + phi.sum():.4f}")
print(f"actual prediction = {pred:.4f}   (match = {abs(base_fmean+phi.sum()-pred)<1e-9})")

## Does the `shap` library agree? (exactly)

`shap` picks a **LinearExplainer** for a linear model. With the same background it returns `base_values = f(mean)` and `values = beta*(x - mean)` — our hand numbers, to the decimal.


In [ ]:
try:
    import shap
except Exception:
    !pip -q install shap
    import shap
explainer = shap.LinearExplainer(lin, xbar.to_frame().T)   # background = the average row (exact for a linear model)
expl = explainer(Xte)
print("row 0 — ours vs shap:")
for j, f in enumerate(FEATURES):
    print(f"  {f:<9} ours {phi[j]:+.4f}   shap {expl.values[0][j]:+.4f}   match={abs(phi[j]-expl.values[0][j])<1e-6}")
print(f"\nshap base value {expl.base_values[0]:.4f}   ours {base_fmean:.4f}")

In [ ]:
# the waterfall IS the regression equation, drawn from the average
shap.plots.waterfall(expl[0])

## Wrap-up

> **Key takeaways**
> - On a **linear** model, SHAP is exact and trivial: $\phi_i=\beta_i(x_i-\bar x_i)$ — *the regression equation, read from the average*.
> - The **base value** is $f(\bar x)$ = the average prediction; **base $+\sum\phi=$ prediction** is just algebra.
> - There are **no coalitions** because a linear model has **no interactions** — every feature adds the same amount regardless of what else is "known".
>
> **Rung 2 (`2a_SHAP_ShallowTree_CAHousing`):** a shallow *tree* has interactions, so a feature's contribution depends on who else is known — and *that* is when you must average over coalitions. Same promise both times: base + contributions = the prediction.
